# Small LLM QA: Few-Shot Retrieval and QLoRA

Run a new experiment with Qwen3-1.7B on an authorized copy of the mixed QA
dataset. This is a clean notebook, not a saved execution transcript. Historical
aggregate results are under `results/` in the repository.

1. Download this repository as ZIP and rename the download `source.zip`.
2. Put `source.zip` and the authorized original `dataset.zip` in Google Drive
   under `MyDrive/small-llm-qa/input/`.
3. Select a **GPU** runtime in Colab. The recorded configuration uses FP16 on
   a Tesla T4; another GPU/driver must pass the checks as a fresh experiment.
4. Run stages 01–06 first, inspect their outputs, then continue in order.

Training and validation may take hours. Runtime availability and exact outputs
are not guaranteed. Save work to Drive. Keep one work directory per fixed
source/configuration; changing either requires a new directory. The data format
and licensing notes are documented in `docs/DATA.md`.


In [ ]:
# 01 — Mount Drive and choose a workspace.
from google.colab import drive
from pathlib import Path
import json, os, shutil, stat, subprocess, sys, tempfile, zipfile

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/small-llm-qa')
INPUT = BASE / 'input'
PROJECT = BASE / 'work' / 'experiment-01'
SOURCE_ARCHIVE = INPUT / 'source.zip'
DATASET_ARCHIVE = INPUT / 'dataset.zip'
for archive in (SOURCE_ARCHIVE, DATASET_ARCHIVE):
    if not archive.is_file():
        raise FileNotFoundError(f'Upload {archive.name} to {INPUT}')
PROJECT.mkdir(parents=True, exist_ok=True)
print('Workspace:', PROJECT)


In [ ]:
# 02 — Extract repository files safely and create a live-output runner.
# Supports both GitHub's top-level archive directory and a flat source ZIP.
with tempfile.TemporaryDirectory(prefix='small-llm-qa-source-') as temporary:
    extracted = Path(temporary)
    with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
        if sum(member.file_size for member in archive.infolist()) > 100 * 1024**2:
            raise ValueError('Source archive exceeds the 100 MiB package limit')
        for member in archive.infolist():
            name = member.filename.replace(chr(92), '/')
            target = (extracted / name).resolve()
            if name.startswith('/') or ':' in name or extracted.resolve() not in target.parents:
                raise ValueError('Unsafe source archive path')
            if stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Source archive must not contain symbolic links')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as destination:
                    shutil.copyfileobj(source, destination)
    roots = [p.parent for p in extracted.rglob('pyproject.toml')
             if (p.parent / 'src' / 'small_llm_qa' / 'workflow.py').is_file()]
    if len(roots) != 1:
        raise ValueError('Expected exactly one complete repository in source.zip')
    for source in roots[0].rglob('*'):
        if not source.is_file():
            continue
        relative = source.relative_to(roots[0])
        if any(part in {'.git', '__pycache__'} or part.startswith('.venv') for part in relative.parts):
            continue
        destination = PROJECT / relative
        content = source.read_bytes()
        if destination.exists() and destination.read_bytes() != content:
            raise RuntimeError('Existing source differs; select a NEW work directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)

os.chdir(PROJECT)
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['HF_HOME'] = '/content/small-llm-qa-hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
DATA = PROJECT / 'data' / 'prepared'
RUNS = PROJECT / 'runs'
RUNS.mkdir(exist_ok=True)
CONFIG = RUNS / 'run_config.json'
FROZEN = RUNS / 'frozen_config.json'
PYTHON = '/content/small-llm-qa-runtime/bin/python'

def run_command(arguments):
    command = list(map(str, arguments))
    process = subprocess.Popen(command, cwd=PROJECT,
        env={**os.environ, 'MPLBACKEND': 'Agg'}, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()
    if result:
        raise subprocess.CalledProcessError(result, command)

def run_module(module, *arguments):
    run_command([PYTHON, '-u', '-m', module, *arguments])

print('Code ready:', PROJECT)


## Runtime and data checks

Stage 03 creates an isolated Python 3.11 runtime with the recorded dependency
pins. These packages were used in the original experiment; installation and
GPU smoke checks must still succeed in a new Colab session. No notebook-kernel
restart is needed because commands use the isolated interpreter.


In [ ]:
# 03 — Install dependencies and inspect the actual GPU.
run_command([sys.executable, '-m', 'pip', 'install', 'uv==0.6.17'])
if not Path(PYTHON).is_file():
    run_command([sys.executable, '-m', 'uv', 'python', 'install', '3.11'])
    run_command([sys.executable, '-m', 'uv', 'venv', '--python', '3.11',
                 '--seed', '/content/small-llm-qa-runtime'])
run_command([PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url',
             'https://download.pytorch.org/whl/cu124'])
run_command([PYTHON, '-m', 'pip', 'install', '-r', PROJECT / 'requirements-colab.txt'])
run_command([PYTHON, '-m', 'pip', 'check'])
run_module('small_llm_qa.workflow', 'environment', '--output', RUNS / 'environment.json')


In [ ]:
# 04 — Prepare answer-blind splits and run CPU checks.
run_module('small_llm_qa.data', '--archive', DATASET_ARCHIVE, '--output', DATA)
print((DATA / 'audit.txt').read_text(encoding='utf-8'))
run_command([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
run_command([PYTHON, PROJECT / 'scripts' / 'demo.py'])


In [ ]:
# 05 — Check every prompt length and assistant-only loss mask.
run_module('small_llm_qa.workflow', 'preflight', '--prepared', DATA,
    '--config', PROJECT / 'configs' / 'experiment.json',
    '--output', RUNS / 'preflight.json', '--gpu')
print(CONFIG.read_text(encoding='utf-8'))


In [ ]:
# 06 — Check real quantized inference on validation examples only.
run_module('small_llm_qa.workflow', 'smoke', '--prepared', DATA,
    '--config', CONFIG, '--output', RUNS / 'smoke_inference')
records = [json.loads(line) for line in
    (RUNS / 'smoke_inference' / 'raw_predictions.jsonl').read_text(encoding='utf-8').splitlines()]
for row in records:
    keys = ['Sample_id', 'Task', 'Answer', 'input_tokens', 'generated_tokens',
            'hit_token_limit', 'generation_mode', 'effective_generation_config']
    print(json.dumps({key: row[key] for key in keys}, indent=2))


Inspect stages 03–06 before continuing. Imports alone do not establish a
working quantized model. Inference should report greedy decoding, sensible
lengths, and no configuration mismatch. If a stage fails, retain its logs
and resolve it before launching long jobs.


In [ ]:
# 07 — Prove a successful optimizer update and nonzero adapter weight change.
# An interrupted/failed smoke needs a fresh smoke output directory.
run_module('small_llm_qa.training', '--train-csv', DATA / 'train.csv',
    '--validation-csv', DATA / 'validation.csv', '--model-config', CONFIG,
    '--output-dir', RUNS / 'smoke_training', '--smoke', '--resume', 'auto')
summary = json.loads((RUNS / 'smoke_training' / 'training_summary.json').read_text(encoding='utf-8'))
keys = ['status', 'global_step', 'training_loss', 'successful_optimizer_updates',
        'amp_skipped_steps', 'adapter_max_weight_delta']
print(json.dumps({key: summary[key] for key in keys}, indent=2))


In [ ]:
# 08 — Evaluate the full validation baseline and choose a common n.
run_module('small_llm_qa.workflow', 'predict', '--prepared', DATA,
    '--config', CONFIG, '--split', 'validation', '--output', RUNS / 'validation_baseline')
run_module('small_llm_qa.workflow', 'tune', '--prepared', DATA,
    '--config', CONFIG, '--output', RUNS / 'tuning')
print((RUNS / 'tuning' / 'selection.json').read_text(encoding='utf-8'))


In [ ]:
# 09 — Train one full epoch, resuming complete checkpoints when available.
run_module('small_llm_qa.training', '--train-csv', DATA / 'train.csv',
    '--validation-csv', DATA / 'validation.csv', '--model-config', CONFIG,
    '--output-dir', RUNS / 'sft', '--resume', 'auto')
from IPython.display import Image, display
display(Image(filename=str(RUNS / 'sft' / 'loss_curve.png')))
summary = json.loads((RUNS / 'sft' / 'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps({key: summary[key] for key in ['status', 'global_step',
    'successful_optimizer_updates', 'amp_skipped_steps', 'training_loss', 'validation']}, indent=2))


## Freeze, predict, then score

Before stage 10, inspect selection evidence and completed training. Test
answers must remain unopened until all four complete prediction sets have
been frozen and sealed. No prompt or checkpoint choice uses test scores.

After a disconnect, rerun 01–05 using exactly the same source and workspace.
Then rerun the interrupted stage. Inference journals and full-training
checkpoints support recovery; inference results are not training checkpoints.
Stage 09 recognizes an already completed compatible training run. Do not
edit a frozen run or use the smoke adapter for final evaluation.


In [ ]:
# 10 — Freeze configuration, validation selection, and completed adapter.
run_module('small_llm_qa.workflow', 'freeze', '--prepared', DATA,
    '--config', CONFIG, '--selection', RUNS / 'tuning' / 'selection.json',
    '--adapter', RUNS / 'sft' / 'adapter', '--output', FROZEN)
print(FROZEN.read_text(encoding='utf-8'))


In [ ]:
# 11 — Save four complete test prediction sets, without reading test answers.
for method in ['baseline', 'random', 'similar', 'sft']:
    run_module('small_llm_qa.workflow', 'predict', '--prepared', DATA,
        '--config', FROZEN, '--method', method, '--split', 'test',
        '--output', RUNS / 'test' / method)
print('Four prediction sets completed; test answers have not been opened.')


In [ ]:
# 12 — Seal prediction paths and checksums.
run_module('small_llm_qa.workflow', 'bundle', '--prepared', DATA,
    '--config', FROZEN, '--runs-dir', RUNS / 'test', '--output', RUNS / 'final_bundle.json')


In [ ]:
# 13 — Open test references for final scoring only.
run_module('small_llm_qa.evaluate', 'final', '--frozen', RUNS / 'final_bundle.json',
    '--answer-archive', DATASET_ARCHIVE, '--output', RUNS / 'final_evaluation.json')
for method in ['baseline', 'random', 'similar', 'sft']:
    print(method, (RUNS / 'test' / method / 'statistics.json').read_text(encoding='utf-8'))


Analyze successes and regressions together. Keep these test results for
explaining this fixed experiment. Further optimization uses validation
data and requires an independent unseen evaluation for new generalization
claims. Generated files remain in your Drive workspace; review notebook
outputs before sharing them, since they may contain private paths or data.
